# Overview

This notebook corresponds to makemore part 3.


In [ ]:
import random

import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Load Data and Build Datasets

Removed checks and EDA (covered in part 2 notebook)


In [ ]:
with open("names.txt", "r") as file:
    names = file.read().splitlines()

In [ ]:
chars = ["."] + sorted(set("".join(names)))
s_to_i = {s: i for i, s in enumerate(chars)}
i_to_s = {i: s for i, s in enumerate(chars)}

vocab_size = len(chars)


def build_dataset(names, block_size=3):
    X, y = [], []
    for name in names:
        context = [0] * block_size
        name += "."
        for ch in name:
            idx = s_to_i[ch]
            X.append(context)
            y.append(idx)
            context = context[1:] + [idx]

    return torch.tensor(X), torch.tensor(y)

In [ ]:
random.seed(42)
random.shuffle(names)

p1, p2 = int(len(names) * 0.8), int(len(names) * 0.9)

block_size = 3

X_train, y_train = build_dataset(names[:p1], block_size)
X_val, y_val = build_dataset(names[p1:p2], block_size)
X_test, y_test = build_dataset(names[p2:], block_size)

# Build MLP


In [ ]:
g = torch.Generator().manual_seed(2147483647)

embedding_dim = 10
hidden_size = 200

C = torch.randn((vocab_size, embedding_dim), generator=g)

# Use Kaiming initialization (tanh version); b1 is just scaled down
W1 = torch.randn((embedding_dim * block_size, hidden_size), generator=g) * (
    (5 / 3) / (embedding_dim * block_size) ** 0.5
)
b1 = torch.randn(hidden_size, generator=g) * 0.01

# Scale down W2 and b2 to prevent extreme logits
W2 = torch.randn((hidden_size, vocab_size), generator=g) * 0.01
b2 = torch.randn(vocab_size, generator=g) * 0

params = [C, W1, b1, W2, b2]
for p in params:
    p.requires_grad = True

print(f"Total parameters: {sum(p.numel() for p in params)}")

# Train and Evaluate MLP


In [ ]:
batch_size = 256
num_steps = 200000
lrs = [0.02, 0.2, 0.02, 0.002]
milestones = [2000, 120000, 150000]

losses = []
j = 0
for i in range(num_steps):
    # Get mini-batch
    idxs = torch.randint(0, X_train.shape[0], (batch_size,), generator=g)
    X_batch, y_batch = X_train[idxs], y_train[idxs]

    # Forward pass
    embs = C[X_batch]
    emb_concat = embs.reshape((len(X_batch), -1))
    preactivation = emb_concat @ W1 + b1
    h = torch.tanh(preactivation)
    logits = h @ W2 + b2

    # Compute loss
    loss = F.cross_entropy(logits, y_batch)

    # Backward pass
    for p in params:
        p.grad = None
    loss.backward()

    # Get learning rate
    if j < len(milestones) and i >= milestones[j]:
        j += 1
    lr = lrs[j]

    # GD step
    with torch.no_grad():
        for p in params:
            p -= lr * p.grad

    # Track loss
    losses.append(loss.item())

    if i % 10000 == 0:
        print(f"Epoch: {i}/{num_steps}, loss: {loss}")

In [ ]:
plt.plot(losses)

In [ ]:
def evaluate(params, X, y):
    C, W1, b1, W2, b2 = params[0], params[1], params[2], params[3], params[4]
    emb = C[X].reshape((len(X), -1))
    h = torch.tanh(emb @ W1 + b1)
    logits = h @ W2 + b2
    return F.cross_entropy(logits, y).item()

In [ ]:
train_loss = evaluate(params, X_train, y_train)
val_loss = evaluate(params, X_val, y_val)

print(f"Train loss: {train_loss}")
print(f"Val loss: {val_loss}")

# Inference


In [ ]:
for _ in range(5):
    x = [0] * block_size
    out = ""
    while True:
        emb = C[torch.tensor(x)].reshape(1, -1)
        h = torch.tanh(emb @ W1 + b1)
        logits = h @ W2 + b2
        probs = F.softmax(logits, dim=1)
        idx = torch.multinomial(probs, num_samples=1).item()

        if idx == 0:
            break

        x = x[1:] + [idx]
        out += i_to_s[idx]

    print(out)